Given a class label, say 207 (golden retriever), generate a 256×256 image.

The approach: define a process that corrupts images into noise over 1000 steps, then train a
network to reverse it. At inference, start from pure noise and run 50 denoising steps.

In [ ]:
%pip install -q matplotlib ipywidgets

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm

torch.manual_seed(0)
torch.set_grad_enabled(False)
device = "cpu"

## Forward process

Each step multiplies the signal by $\sqrt{1-\beta_t}$ and adds $\sqrt{\beta_t}$ of fresh Gaussian noise:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \sqrt{1-\beta_t}\,\mathbf{x}_{t-1} + \sqrt{\beta_t}\,\boldsymbol{\varepsilon}$$

$\beta_t$ runs linearly from $0.0001 \to 0.02$. By $t = 999$ the image is indistinguishable from noise.

The closed form lets you jump directly to any timestep:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\varepsilon}, \qquad \bar{\alpha}_t = \prod_{i=1}^{t}(1-\beta_i)$$

In [ ]:
T              = 1000
betas          = np.linspace(0.0001, 0.02, T, dtype=np.float64)
alphas         = 1.0 - betas
alphas_cumprod = np.cumprod(alphas)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(betas * 1e3, color='steelblue', lw=2)
ax1.set(title=r'$\beta_t$ (noise per step, ×10⁻³)', xlabel='t')
ax2.plot(alphas_cumprod, color='darkorange', lw=2)
ax2.axhline(0.5, ls='--', color='gray', alpha=0.6, label='50% signal')
ax2.set(title=r'$\bar{\alpha}_t$ (signal remaining)', xlabel='t')
ax2.legend()
plt.tight_layout(); plt.show()
print(f'signal at t=500: {alphas_cumprod[500]:.3f}   at t=999: {alphas_cumprod[999]:.5f}')

In [ ]:
import os
os.makedirs('assets', exist_ok=True)
!wget -q https://raw.githubusercontent.com/alexhamidi/alexhamidi.github.io/main/public/projects/t.png -O assets/t.png

In [ ]:
img_pil    = Image.open('assets/t.png').convert('RGB').resize((256, 256), Image.LANCZOS)
img_tensor = torch.from_numpy(np.array(img_pil, dtype=np.float32) / 255.0).permute(2, 0, 1)

torch.manual_seed(1)
noise = torch.randn_like(img_tensor)

fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
for ax, t, lbl in zip(axes, [0,200,500,800,999], ['t=0','t=200','t=500','t=800','t=999']):
    ab    = float(alphas_cumprod[t])
    noisy = (np.sqrt(ab) * img_tensor + np.sqrt(1 - ab) * noise).clamp(0, 1)
    ax.imshow(noisy.permute(1, 2, 0).numpy())
    ax.set_title(lbl, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()

## Reverse process

To generate, start from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0},\mathbf{I})$ and sample backward.

The true reverse distribution requires $p(\mathbf{x}_0)$, the full data distribution, which we
don't have. Instead, a network predicts $\hat{\mathbf{x}}_0$ from $(\mathbf{x}_t, t, \text{label})$,
then we sample from the posterior $q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \hat{\mathbf{x}}_0)$, which is tractable.

### Posterior derivation

Apply Bayes' rule, then use the Markov property:

$$q(\mathbf{x}_{t-1}\mid\mathbf{x}_t, \hat{\mathbf{x}}_0)
= \frac{q(\mathbf{x}_t\mid\mathbf{x}_{t-1})\; q(\mathbf{x}_{t-1}\mid\hat{\mathbf{x}}_0)}{q(\mathbf{x}_t\mid\hat{\mathbf{x}}_0)}$$

Every term is a Gaussian. Expanding and collecting terms in $\mathbf{x}_{t-1}$ gives a Gaussian kernel:

$$\propto \exp\!\left(-\frac{1}{2}\left[
\mathbf{x}_{t-1}^2\underbrace{\left(\frac{\alpha_t}{\beta_t}+\frac{1}{1-\bar{\alpha}_{t-1}}\right)}_{1/\tilde{\beta}_t}
- 2\mathbf{x}_{t-1}\underbrace{\left(\frac{\sqrt{\alpha_t}}{\beta_t}\mathbf{x}_t+\frac{\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_{t-1}}\hat{\mathbf{x}}_0\right)}_{\tilde{\mu}_t/\tilde{\beta}_t}
\right]\right)$$

Reading off the variance and mean:

$$\tilde{\beta}_t = \beta_t\cdot\frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}
\qquad
\tilde{\mu}_t = \frac{\beta_t\sqrt{\bar{\alpha}_{t-1}}}{1-\bar{\alpha}_t}\hat{\mathbf{x}}_0
             + \frac{(1-\bar{\alpha}_{t-1})\sqrt{\alpha_t}}{1-\bar{\alpha}_t}\mathbf{x}_t$$

$$\mathbf{x}_{t-1} = \tilde{\mu}_t + \sqrt{\tilde{\beta}_t}\,\boldsymbol{\varepsilon}$$

The network also predicts variance. It outputs $v \in [-1,1]$ that interpolates between a lower
bound $\log\tilde{\beta}_t$ and upper bound $\log\beta_t$ (Nichol & Dhariwal 2021).

At inference we use 50 evenly-spaced steps instead of 1000. Skipping steps changes the effective
$\beta$ for each jump: $\tilde{\beta} = 1 - \bar{\alpha}_t / \bar{\alpha}_{t_{\text{prev}}}$.
All coefficients below are computed from this spaced schedule.

In [ ]:
used_timesteps      = np.linspace(0, 999, 50, dtype=int)

signal_used         = alphas_cumprod[used_timesteps]
signal_prev         = np.concatenate([[1.0], signal_used[:-1]])
spaced_betas        = 1 - signal_used / signal_prev
spaced_alphas       = 1 - spaced_betas
spaced_ac           = np.cumprod(signal_used / signal_prev)
spaced_ac_prev      = np.concatenate([[1.0], spaced_ac[:-1]])

spaced_sqrt_recip   = np.sqrt(1.0 / spaced_ac)
spaced_sqrt_recipm1 = np.sqrt(1.0 / spaced_ac - 1)
spaced_post_coef1   = spaced_betas * np.sqrt(spaced_ac_prev) / (1.0 - spaced_ac)
spaced_post_coef2   = (1.0 - spaced_ac_prev) * np.sqrt(spaced_alphas) / (1.0 - spaced_ac)
spaced_post_var     = spaced_betas * (1.0 - spaced_ac_prev) / (1.0 - spaced_ac)
spaced_post_log_var = np.log(np.append(spaced_post_var[1], spaced_post_var[1:]))
spaced_log_betas    = np.log(spaced_betas)

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.plot(alphas_cumprod, color='darkorange', lw=2, alpha=0.4, label='all 1000 steps')
ax.plot(used_timesteps, alphas_cumprod[used_timesteps], 'o', color='steelblue', ms=5, label='50 used')
ax.set(xlabel='t', ylabel=r'$\bar{\alpha}_t$')
ax.legend(); plt.tight_layout(); plt.show()

## The network

The network takes $(\mathbf{x}_t, t, \text{label})$ and predicts the noise $\boldsymbol{\varepsilon}$
and variance $v$. It's a transformer that operates on patches of the latent.

The VAE from Stable Diffusion first compresses images from $256{\times}256{\times}3$ to $32{\times}32{\times}4$.
Working in latent space is cheaper and produces better samples than pixel-space diffusion.

### Patching

The $4{\times}32{\times}32$ latent is split into non-overlapping $2{\times}2$ patches.
That gives $16 \times 16 = 256$ patches, each containing $4 \times 4 = 16$ values.
These 256 patches become the 256 tokens the transformer processes.

In [ ]:
torch.manual_seed(42)
latent = torch.randn(4, 32, 32)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(latent[0].numpy(), cmap='RdBu', vmin=-2.5, vmax=2.5)
axes[0].set_title('latent channel 0  (4×32×32)', fontsize=10); axes[0].axis('off')

axes[1].imshow(latent[0].numpy(), cmap='RdBu', vmin=-2.5, vmax=2.5)
for v in range(0, 32, 2):
    axes[1].axvline(v - 0.5, color='white', lw=0.6, alpha=0.7)
    axes[1].axhline(v - 0.5, color='white', lw=0.6, alpha=0.7)
axes[1].set_title('2×2 patch grid  →  256 patches', fontsize=10); axes[1].axis('off')

patch_means = latent[0].unfold(0,2,2).unfold(1,2,2).reshape(16,16,4).mean(-1)
axes[2].imshow(patch_means.numpy(), cmap='RdBu', vmin=-1.5, vmax=1.5)
axes[2].set_title('each patch = one token  (16×16)', fontsize=10); axes[2].axis('off')

plt.tight_layout(); plt.show()

### Patch embedding

`PatchEmbed` projects each 16-value patch to 1152 dimensions. Internally it's a single strided
convolution: `Conv2d(in_channels=4, out_channels=1152, kernel_size=2, stride=2)`, sliding over
the latent one patch at a time. Output: 256 tokens x 1152 dims.

In [ ]:
from timm.models.vision_transformer import PatchEmbed

x_embedder = PatchEmbed(img_size=32, patch_size=2, in_chans=4, embed_dim=1152)

dummy = torch.randn(1, 4, 32, 32)
tokens = x_embedder(dummy)
print(f'latent {list(dummy.shape)}  →  tokens {list(tokens.shape)}')

### Positional encoding

After projection, patches don't know where they are in the image. We fix this by adding a fixed
2D sinusoidal encoding to each token.

The 1152D encoding splits in half: 576 dims encode row position, 576 encode column position.
Within each half, different dimensions use different frequencies. Low frequencies encode coarse
position, high frequencies encode fine position. Every (row, col) pair maps to a unique 1152D vector.

In [ ]:
grid_size = 16
D         = 288
freqs     = 1.0 / (10000 ** (np.arange(D) / D))

fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))
for ax, d in zip(axes.flat, [0, 2, 8, 30, 100, 250]):
    rows = np.sin(np.arange(grid_size) * freqs[d])
    cols = np.cos(np.arange(grid_size) * freqs[d])
    ax.imshow(np.outer(rows, cols), cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f'dim {d}  (freq {freqs[d]:.4f})', fontsize=8.5)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('6 of 576 row×col frequency components',
             fontsize=9, y=1.02)
plt.tight_layout(); plt.show()

### Timestep embedding

The network needs to know the noise level. Timestep $t$ (a scalar 0–999) is encoded as a 256D
sinusoidal vector, then expanded to 1152D by a small MLP:

$$\text{enc}(t)_d = \cos(t \cdot \omega_d),\; \sin(t \cdot \omega_d)$$

Different $t$ values produce different patterns. The plot shows this for four timesteps.

In [ ]:
class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    def forward(self, t):
        half  = self.frequency_embedding_size // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, dtype=torch.float32) / half
        ).to(t.device)
        args   = t[:, None].float() * freqs[None]
        t_freq = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return self.mlp(t_freq)

In [ ]:
half  = 128
freqs = np.exp(-np.log(10000) * np.arange(half) / half)

fig, ax = plt.subplots(figsize=(12, 3))
for t, c in [(0,'steelblue'),(100,'darkorange'),(500,'green'),(999,'crimson')]:
    enc = np.concatenate([np.cos(t * freqs), np.sin(t * freqs)])
    ax.plot(enc[:80], lw=1.5, color=c, alpha=0.85, label=f't={t}')
ax.set(xlabel='dimension', title='sinusoidal timestep encoding (first 80 of 256 dims)')
ax.axhline(0, color='gray', lw=0.5); ax.legend()
plt.tight_layout(); plt.show()

### Label embedding

A lookup table maps class index $y$ to 1152D. It has 1001 rows. The extra row at index 1000
is the unconditional embedding used during classifier-free guidance.

During training, the model randomly replaces labels with row 1000 at 10% rate. This teaches it
to generate without a class signal, which makes CFG possible at inference.

In [ ]:
class LabelEmbedder(nn.Module):
    def __init__(self, num_classes, hidden_size, dropout_prob):
        super().__init__()
        self.embedding_table = nn.Embedding(num_classes + (dropout_prob > 0), hidden_size)
        self.num_classes  = num_classes
        self.dropout_prob = dropout_prob

    def forward(self, labels, train, force_drop_ids=None):
        if (train and self.dropout_prob > 0) or force_drop_ids is not None:
            drop_ids = (
                torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
                if force_drop_ids is None else force_drop_ids == 1
            )
            labels = torch.where(drop_ids, self.num_classes, labels)
        return self.embedding_table(labels)

### DiT block

Each block is a standard transformer block (self-attention + MLP) with one change: the layer norms
are conditioned on $\mathbf{c} = \mathbf{t}_{\text{emb}} + \mathbf{y}_{\text{emb}}$ via adaLN.

A linear layer projects $\mathbf{c}$ to 6 scalars that modulate both sublayers:

- `shift`, `scale` modify the LayerNorm output: $x \cdot (1 + \text{scale}) + \text{shift}$
- `gate` scales the residual: $x \leftarrow x + \text{gate} \cdot \text{sublayer}(x)$

The final linear in `adaLN_modulation` is zero-initialized, so each block starts as an identity.
Conditioning turns on gradually during training.

In [ ]:
from timm.models.vision_transformer import Attention, Mlp

class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0, **block_kwargs):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn  = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, **block_kwargs)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.mlp   = Mlp(
            in_features=hidden_size,
            hidden_features=int(hidden_size * mlp_ratio),
            act_layer=lambda: nn.GELU(approximate='tanh'),
            drop=0,
        )
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True),  # zero-init
        )

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, \
        shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6, dim=1)

        x = x + gate_msa.unsqueeze(1) * self.attn(
            self.norm1(x) * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)
        )
        x = x + gate_mlp.unsqueeze(1) * self.mlp(
            self.norm2(x) * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)
        )
        return x

### Final layer

After 28 blocks, tokens are still 1152D representations. The final layer applies adaLN once more
and projects $1152 \to 32$ per token: $\text{patch\_size}^2 \times 8 = 4 \times 8 = 32$.
First 4 channels are the noise prediction $\boldsymbol{\varepsilon}_\theta$, last 4 are variance $v$.

In [ ]:
class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear     = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True),
        )

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = self.norm_final(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        return self.linear(x)

### Full model

`forward(x, t, y)`: patchify → embed → add positional encoding → compute
$\mathbf{c} = \mathbf{t}_{\text{emb}} + \mathbf{y}_{\text{emb}}$ → 28 blocks → final layer → unpatchify.

`forward_with_cfg` runs conditional and unconditional in one batch, then blends:

$$\boldsymbol{\varepsilon} = \boldsymbol{\varepsilon}_{\text{uncond}} + s\cdot(\boldsymbol{\varepsilon}_{\text{cond}} - \boldsymbol{\varepsilon}_{\text{uncond}})$$

Higher $s$ = stronger class signal, less diversity.

In [ ]:
class DiT(nn.Module):
    def __init__(self, img_size=32, patch_size=2, in_channels=4,
                 hidden_size=1152, depth=28, num_heads=16, mlp_ratio=4.0,
                 class_dropout_prob=0.1, num_classes=1000, learn_sigma=True):
        super().__init__()
        self.learn_sigma  = learn_sigma
        self.in_channels  = in_channels
        self.out_channels = in_channels * 2 if learn_sigma else in_channels
        self.patch_size   = patch_size
        self.num_heads    = num_heads

        self.x_embedder = PatchEmbed(img_size, patch_size, in_channels, hidden_size, bias=True)
        self.t_embedder = TimestepEmbedder(hidden_size)
        self.y_embedder = LabelEmbedder(num_classes, hidden_size, class_dropout_prob)
        self.pos_embed  = nn.Parameter(
            torch.zeros(1, self.x_embedder.num_patches, hidden_size), requires_grad=False
        )
        self.blocks      = nn.ModuleList([
            DiTBlock(hidden_size, num_heads, mlp_ratio=mlp_ratio) for _ in range(depth)
        ])
        self.final_layer = FinalLayer(hidden_size, patch_size, self.out_channels)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
        embed_dim = self.pos_embed.shape[-1]
        grid_size = int(self.x_embedder.num_patches ** 0.5)
        grid      = np.stack(np.meshgrid(np.arange(grid_size, dtype=np.float32),
                                          np.arange(grid_size, dtype=np.float32)), axis=0
                             ).reshape([2, 1, grid_size, grid_size])
        half  = embed_dim // 2
        omega = 1.0 / 10000 ** (np.arange(half//2, dtype=np.float64) / (half/2.0))
        def _enc(x): return np.concatenate(
            [np.sin(np.einsum('m,d->md', x.reshape(-1), omega)),
             np.cos(np.einsum('m,d->md', x.reshape(-1), omega))], axis=1)
        self.pos_embed.data.copy_(
            torch.from_numpy(np.concatenate([_enc(grid[0]), _enc(grid[1])], axis=1)).float().unsqueeze(0)
        )
        w = self.x_embedder.proj.weight.data
        nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        nn.init.constant_(self.x_embedder.proj.bias, 0)
        nn.init.normal_(self.y_embedder.embedding_table.weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[2].weight, std=0.02)
        for block in self.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias,   0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].bias,   0)
        nn.init.constant_(self.final_layer.linear.weight, 0)
        nn.init.constant_(self.final_layer.linear.bias,   0)

    def forward(self, x, t, y):
        x = self.x_embedder(x) + self.pos_embed
        t = self.t_embedder(t)
        y = self.y_embedder(y, self.training)
        c = t + y
        for block in self.blocks:
            x = block(x, c)
        x = self.final_layer(x, c)
        p = self.x_embedder.patch_size[0]
        h = w = int(x.shape[1] ** 0.5)
        x = x.reshape(x.shape[0], h, w, p, p, self.out_channels)
        x = torch.einsum('nhwpqc->nchpwq', x)
        return x.reshape(x.shape[0], self.out_channels, h * p, h * p)

    def forward_with_cfg(self, x, t, y, cfg_scale):
        half      = x[: len(x) // 2]
        combined  = torch.cat([half, half], dim=0)
        model_out = self.forward(combined, t, y)
        eps, rest            = model_out[:, :self.in_channels], model_out[:, self.in_channels:]
        cond_eps, uncond_eps = torch.split(eps, len(eps) // 2, dim=0)
        half_eps             = uncond_eps + cfg_scale * (cond_eps - uncond_eps)
        return torch.cat([torch.cat([half_eps, half_eps], dim=0), rest], dim=1)

### Loading weights

DiT-XL/2 was trained on ImageNet 256×256 for 7M steps. We load the EMA weights,
which are smoother than the raw training weights and produce better samples.

In [ ]:
from torchvision.datasets.utils import download_url

IMAGE_SIZE  = 256
LATENT_SIZE = IMAGE_SIZE // 8
IN_CHANNELS = 4
NUM_CLASSES = 1000

ckpt_name  = f'DiT-XL-2-{IMAGE_SIZE}x{IMAGE_SIZE}.pt'
local_path = f'pretrained_models/{ckpt_name}'

if not os.path.isfile(local_path):
    os.makedirs('pretrained_models', exist_ok=True)
    download_url(f'https://dl.fbaipublicfiles.com/DiT/models/{ckpt_name}', 'pretrained_models')

state_dict = torch.load(local_path, map_location='cpu')
if 'ema' in state_dict: state_dict = state_dict['ema']

model = DiT(img_size=LATENT_SIZE, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(state_dict)
model.eval()
print(f'{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters')

The actual positional embeddings from the loaded model, reshaped to the 16×16 patch grid:

In [ ]:
pe = model.pos_embed.data[0].numpy()   # (256, 1152)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for i, ax in enumerate(axes):
    ax.imshow(pe[:, i * 280].reshape(16, 16), cmap='RdBu')
    ax.set_title(f'dim {i*280}', fontsize=9); ax.axis('off')
plt.suptitle('positional embeddings, 16x16 patch grid, 4 dimensions shown', fontsize=9, y=1.01)
plt.tight_layout(); plt.show()

## VAE decoder

The model outputs a $4{\times}32{\times}32$ noise prediction in latent space. To get a viewable image
we decode with the Stable Diffusion VAE: $4{\times}32{\times}32 \to 3{\times}256{\times}256$.
VAE training scales latents by $0.18215$, so we divide by that before decoding.

In [ ]:
from diffusers.models import AutoencoderKL
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(device)

## Sampling

Set class, CFG scale, and seed below. Classes to try: `207` golden retriever, `980` volcano,
`388` giant panda, `130` flamingo, `291` lion, `417` balloon.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    CLASSES = {207:'golden retriever', 980:'volcano', 388:'giant panda',
               291:'lion', 130:'flamingo', 417:'balloon', 985:'daisy', 850:'teddy bear'}
    w_class = widgets.Dropdown(
        options=sorted([(f'{k}  {v}', k) for k, v in CLASSES.items()]),
        value=207, description='class:', style={'description_width':'initial'},
        layout=widgets.Layout(width='260px'))
    w_cfg = widgets.FloatSlider(
        value=4.0, min=1.0, max=10.0, step=0.5, description='CFG scale:',
        style={'description_width':'initial'}, layout=widgets.Layout(width='330px'))
    w_seed = widgets.IntSlider(
        value=9, min=0, max=99, description='seed:',
        style={'description_width':'initial'}, layout=widgets.Layout(width='260px'))
    display(w_class, w_cfg, w_seed)
except ImportError:
    w_class = type('W', (), {'value': 207})()
    w_cfg   = type('W', (), {'value': 4.0})()
    w_seed  = type('W', (), {'value': 9})()

In [ ]:
CLASS_LABEL = w_class.value
CFG_SCALE   = w_cfg.value
SEED        = w_seed.value
NUM_STEPS   = 50
print(f'class {CLASS_LABEL}  CFG {CFG_SCALE}  seed {SEED}')

In [ ]:
torch.manual_seed(SEED)
noise_tensor = torch.randn(1, IN_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device)
img = torch.cat([noise_tensor, noise_tensor], dim=0)           # (2, 4, 32, 32)
y   = torch.tensor([CLASS_LABEL, NUM_CLASSES], device=device)  # conditional + unconditional

Each step: run the model → recover $\hat{\mathbf{x}}_0$ → compute posterior mean → sample $\mathbf{x}_{t-1}$.
No noise is added on the last step.

In [ ]:
snapshots = {}
CAPTURE_AT = {49, 39, 29, 19, 9, 0}

for i in tqdm(reversed(range(NUM_STEPS))):
    t_batch = torch.tensor([used_timesteps[i]] * img.shape[0], device=device)

    model_out        = model.forward_with_cfg(img, t_batch, y=y, cfg_scale=CFG_SCALE)
    model_eps        = model_out[:, :IN_CHANNELS]
    model_var_values = model_out[:, IN_CHANNELS:]

    frac         = (model_var_values + 1) / 2
    log_variance = frac * float(spaced_log_betas[i]) + (1 - frac) * float(spaced_post_log_var[i])

    pred_xstart = float(spaced_sqrt_recip[i]) * img - float(spaced_sqrt_recipm1[i]) * model_eps
    mean        = float(spaced_post_coef1[i]) * pred_xstart + float(spaced_post_coef2[i]) * img

    noise = torch.randn_like(img)
    img   = mean + (0.0 if i == 0 else 1.0) * torch.exp(0.5 * log_variance) * noise

    if i in CAPTURE_AT:
        ch0 = img[0, 0].cpu().float()
        snapshots[i] = ((ch0 - ch0.min()) / (ch0.max() - ch0.min() + 1e-8)).numpy()

In [ ]:
fig, axes = plt.subplots(1, len(snapshots), figsize=(14, 2.8))
for ax, i in zip(axes, sorted(snapshots.keys(), reverse=True)):
    ax.imshow(snapshots[i], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f't={used_timesteps[i]}', fontsize=10); ax.axis('off')
plt.suptitle('latent channel 0  (left=noisy, right=clean)', y=1.02, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
from IPython.display import display as ipy_display
from torchvision.transforms.functional import to_pil_image

samples = img.chunk(2, dim=0)[0]
samples = vae.decode(samples / 0.18215).sample
samples = (samples.clamp(-1, 1) + 1) / 2
ipy_display(to_pil_image(samples[0].cpu()))